# E-commerce Sales Analysis
**Dataset:** 10,000 synthetic e-commerce orders (practice dataset)  
**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Goal:** Clean the data, find what drives revenue and returns, and turn charts into business insights.

> Google Colab: upload `ecommerce_orders_10k.csv` from the left Files panel first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("ecommerce_orders_10k.csv")
df.head()

## 1. First look at the data

In [ ]:
print(df.shape)
df.info()
print(df.isna().sum())
df.describe()

## 2. Data cleaning
Order matters: fix the problems first, *then* fill the nulls.
1. Remove duplicates  2. Fix city names  3. Invalid ages -> NaN  4. Fill nulls  5. Remove impossible orders  6. Convert dates

In [ ]:
# 1. duplicates
df = df.drop_duplicates()

# 2. messy city names ("mumbai ", "BENGALURU")
df["city"] = df["city"].str.strip().str.title()
print("cities:", df["city"].nunique())

# 3. impossible ages -> NaN
df["customer_age"] = pd.to_numeric(df["customer_age"], errors="coerce")
df.loc[df["customer_age"] > 100, "customer_age"] = np.nan

# 4. fill nulls
df["customer_age"] = df["customer_age"].fillna(df["customer_age"].median())  # median: robust to outliers
df["payment_method"] = df["payment_method"].fillna("Unknown")                  # 'Unknown' keeps the row
# rating left as NaN on purpose: we do not guess customer opinions

# 5. order_amount can never exceed unit_price * quantity (discount only reduces it)
gross = df["unit_price"] * df["quantity"]
print("invalid orders removed:", (df["order_amount"] > gross + 1).sum())
df = df[df["order_amount"] <= gross + 1]

# 6. dates and new columns
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df["weekday"] = df["order_date"].dt.day_name()
df["is_returned"] = (df["returned"] == "Yes").astype(int)

print(df.shape)
print(df.isna().sum())

## 3. Exploratory analysis

In [ ]:
cat = df.groupby("category")["order_amount"].agg(["sum", "count", "mean"]).sort_values("sum", ascending=False)
cat["share_%"] = (cat["sum"] / cat["sum"].sum() * 100).round(1)
cat

In [ ]:
city = df.groupby("city")["order_amount"].agg(["sum", "count", "mean"]).sort_values("sum", ascending=False)
city

In [ ]:
month = df.groupby("month")["order_amount"].agg(["sum", "count"])
month

## 4. Charts

In [ ]:
cat["sum"].plot(kind="bar")
plt.title("Revenue by Category"); plt.ylabel("Revenue"); plt.show()

In [ ]:
month["sum"].plot(kind="line", marker="o")
plt.title("Monthly Revenue"); plt.xlabel("Month"); plt.ylabel("Revenue"); plt.show()

In [ ]:
city["sum"].sort_values().plot(kind="barh")
plt.title("Revenue by City"); plt.show()

In [ ]:
rate = df.groupby("payment_method")["is_returned"].mean() * 100
rate.sort_values().plot(kind="barh")
plt.title("Return Rate % by Payment Method"); plt.show()

In [ ]:
num = df[["unit_price", "quantity", "discount_pct", "order_amount", "delivery_days", "rating", "is_returned"]]
sns.heatmap(num.corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Correlation"); plt.show()

## 5. Insights
*Format: what the data shows (with numbers) -> why -> what the business should do.*

**Category:**  
*(your insight here)*

**Month:**  
*(your insight here)*

**City:**  
*(your insight here)*

**Returns:**  
*(your insight here)*

## 6. Conclusion & next steps
*(2-3 lines in your own words)*

Next: predict order returns with a machine learning model.